# 02 - Data Cleaning & Feature Engineering
## RHoMIS Analytics — Section C, Group 7

---

This notebook focuses on structural cleaning only — column selection, unit standardisation, data entry correction, categorical validation, and household size derivation. No KPIs are computed here. Food Security Status, Income Per Capita, and Farm Size Bucket belong in notebooks 03/04.


In [35]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


---
## 1. Load Data & Column Reduction
We drop the 1,400+ highly sparse columns and isolate only our focused domain indicators.

In [ ]:
raw_path = '../data/raw/raw.csv.gz'
columns_to_keep = [
    # Identity & Geography
    'id_unique', 'country', 'iso_country_code', 'year', 'region',
    'gps_lat_rounded', 'gps_lon_rounded',
    # Household Demographics
    'respondentsex', 'age_malehead', 'age_femalehead', 'education_level',
    'respondent_is_head', 'children_under_4', 'children_4to10', 'males11to24', 'females11to24',
    'males25to50', 'females25to50', 'malesover50', 'femalesover50', 'count_people',
    # Farm & Land
    'landcultivated', 'unitland', 'land_ownership', 'land_tenure', 'land_irrigated', 'farm_labour',
    # Crop Diversity & Production
    'crop_count', 'crop_name_1', 'crop_name_2', 'crop_name_3', 'crop_name_4', 'crop_name_5',
    'crop_harvest_kg_per_year_1', 'crop_harvest_kg_per_year_2', 'crop_harvest_kg_per_year_3',
    'crop_income_per_year_1', 'crop_income_per_year_2', 'crop_income_per_year_3',
    'crop_land_area_1', 'crop_land_area_2', 'crop_land_area_3',
    'crop_consumed_prop_1', 'crop_consumed_prop_2', 'crop_consumed_prop_3',
    # Income
    'local_currency', 'offfarm_incomes_any', 'offfarm_income_proportion',
    'livestock_sale_income_1', 'livestock_sale_income_2',
    # Food Security
    'hfias_1', 'hfias_2', 'hfias_3', 'hfias_4', 'hfias_5',
    'hfias_6', 'hfias_7', 'hfias_8', 'hfias_9',
    'fies_1', 'fies_2', 'fies_3', 'fies_4', 'fies_5',
    'fies_6', 'fies_7', 'fies_8',
    'foodshortagetime',
    # Gender & Resource Control
    'crop_who_control_revenue_1', 'crop_who_control_revenue_2', 'crop_who_control_revenue_3',
    'offfarm_who_control_revenue_1', 'offfarm_who_control_revenue_2',
    'livestock_meat_who_control_eating_1', 'dairy_products_who_control_eating',
]
df = pd.read_csv(raw_path, low_memory=False, usecols=columns_to_keep)


---
## 2. Demographic Standardization & Outlier Management
We standardize categorical inputs for sex, correct ages entered incorrectly as birth years, and derive household size from age-band columns.

In [38]:
# Global string cleaning: strip whitespace, lowercase, convert blank strings to NaN
str_cols = df.select_dtypes(include=["object"]).columns
for col in str_cols:
    df[col] = df[col].str.strip().str.lower().replace("", np.nan)

# Standardise country names: lowercase, strip, fix misspellings
df['country'] = df['country'].replace({
    'philipines': 'philippines',
    'philipines ': 'philippines'
})

# Show standardisation result

# Standardise local_currency: map CFA variants to canonical form
currency_map = {
    'cfa_franc': 'xof',
    'fcfa_xof': 'xof',
    'fcfa': 'xof',
    'cfa': 'xof',
    '1': np.nan  # Garbage entry
}

df['local_currency'] = df['local_currency'].replace(currency_map)



In [39]:
def fix_age_from_survey_year(row, age_col):
    val = row[age_col]
    survey_year = row['year']
    
    if pd.isna(val) or pd.isna(survey_year):
        return np.nan
    
    try:
        val = float(val)
        # If it's a birth year (1900 < val < survey_year), convert to age
        if 1900 < val < survey_year:
            age = survey_year - val
            # After conversion, any age >110 is irrecoverable garbage
            return age if age <= 110 else np.nan
        # If it's already an age (0 < val < 110), keep it
        elif 0 < val <= 110:
            return val
        else:
            return np.nan
    except (ValueError, TypeError):
        return np.nan

# Apply age correction
df['age_malehead'] = df.apply(lambda row: fix_age_from_survey_year(row, 'age_malehead'), axis=1)
df['age_femalehead'] = df.apply(lambda row: fix_age_from_survey_year(row, 'age_femalehead'), axis=1)


In [40]:
age_band_cols = [
    'children_under_4', 'children_4to10', 'males11to24', 'females11to24',
    'males25to50', 'females25to50', 'malesover50', 'femalesover50'
]

# Coerce age-band columns to numeric before summing
for col in age_band_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Verify all age-band columns exist and are numeric
for col in age_band_cols:
    if col in df.columns:
        missing_in_col = df[col].isnull().sum()
        if missing_in_col > 0:
            print(f"  ⚠ {col}: {missing_in_col} missing values")

# Create derived household size (sum of all age bands)
df['household_size_derived'] = df[age_band_cols].sum(axis=1)

# Retain original count_people as-is (with NaN intact) for secondary verification
# Do NOT impute count_people — this is per the imputation policy


In [41]:
unitland_conversion = {
    'ha': 1.0,
    'hectare': 1.0,
    'hectares': 1.0,
    'acre': 0.404686,
    'acres': 0.404686,
    'timad': 0.25,
    'timad_0.25ha': 0.25,
    'are': 0.01,
    'ares': 0.01,
    'are_25x25m': 0.01,
    'carre_25x25m': 0.01,
    'm2': 0.0001,
    'metre_metre': 0.0001,
    'm_sq': 0.0001,
    'manzana': 0.7,
    'dunum_1000m2': 0.1,
    'sao': 0.036,
    'saos': 0.036,
    'bigha': 0.677,
    'katha': 0.034,
    'kattha': 0.034,
    'nali': 0.02,
    'naali': 0.02,
    'nalies': 0.02,
    'lima': 0.09,
    'igito_60x60m': 0.36,
    'kanal': 0.05,
    'cuadra_7056msq': 0.7056
}

def convert_land_to_hectares(row):
    val = row['landcultivated']
    unit = str(row['unitland']).lower().strip() if pd.notna(row['unitland']) else ''
    
    if pd.isna(val):
        return np.nan
    
    try:
        val = float(val)
    except (ValueError, TypeError):
        return np.nan
    
    if unit in unitland_conversion:
        return val * unitland_conversion[unit]
    else:
        return np.nan

# Coerce landcultivated to numeric before conversion
df['landcultivated'] = pd.to_numeric(df['landcultivated'], errors='coerce')

# Apply conversion
df['land_cultivated_ha'] = df.apply(convert_land_to_hectares, axis=1)




In [43]:
def standardise_binary_field(series, col_name):
    
    def map_to_standard(val):
        if pd.isna(val):
            return np.nan
        val_lower = str(val).lower().strip()
        if val_lower in ('y', 'yes', 'true', '1'):
            return 'y'
        elif val_lower in ('n', 'no', 'false', '0'):
            return 'n'
        else:
            return np.nan  # no_answer, dont_know, etc. → Nan
    
    return series.apply(map_to_standard)

# Apply standardisation to binary fields
binary_fields = ['offfarm_incomes_any', 'respondent_is_head', 'land_irrigated', 'foodshortagetime']

for col in binary_fields:
    if col in df.columns:
        original_unique = df[col].unique()
        df[col] = standardise_binary_field(df[col], col)

# Special handling for respondentsex (should be 'm'/'f', flag garbage)
if 'respondentsex' in df.columns:
    
    def standardise_sex(val):
        if pd.isna(val):
            return np.nan
        val_lower = str(val).lower().strip()
        if val_lower in ('m', 'male'):
            return 'm'
        elif val_lower in ('f', 'female', 'woman'):
            return 'f'
        else:
            # 'other', 'false', etc. are irrecoverable — set to NaN
            return np.nan
    
    df['respondentsex'] = df['respondentsex'].apply(standardise_sex)


In [45]:
allowed_crop_land_area = {'little', 'underhalf', 'half', 'most', 'all', 'none'}

def validate_crop_land_area(val):
    if pd.isna(val):
        return np.nan
    val_lower = str(val).lower().strip()
    if val_lower in allowed_crop_land_area:
        return val_lower
    else:
        return np.nan 
crop_area_cols = ['crop_land_area_1', 'crop_land_area_2', 'crop_land_area_3']

for col in crop_area_cols:
    if col in df.columns:
        numeric_entries_before = df[col].notna().sum()
        df[col] = df[col].apply(validate_crop_land_area)
        numeric_entries_after = df[col].notna().sum()
        removed = numeric_entries_before - numeric_entries_after
        

# Validate crop_consumed_prop columns as categoricals
consumed_prop_ok = {"none", "little", "underhalf", "half", "most", "all"}
for col in ['crop_consumed_prop_1', 'crop_consumed_prop_2', 'crop_consumed_prop_3']:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda v: v if pd.notna(v) and str(v).strip().lower() in consumed_prop_ok else np.nan
        )

# Coerce numeric columns
for col in ["crop_count", "gps_lat_rounded", "gps_lon_rounded", "count_people"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')


In [ ]:
hfias_allowed = {'never', 'monthly', 'fewpermonth', 'weekly', 'fewperweek', 'daily', 'no_answer'}
fies_allowed = {'y', 'n', 'yes', 'no', 'no_answer'}

def clean_hfias_string(val):
    if pd.isna(val):
        return np.nan
    val_lower = str(val).lower().strip()
    if val_lower in hfias_allowed:
        return val_lower
    else:
        return np.nan

def clean_fies_string(val):
    if pd.isna(val):
        return np.nan
    val_lower = str(val).lower().strip()
    if val_lower in ('yes', 'y'):
        return 'y'
    elif val_lower in ('no', 'n'):
        return 'n'
    elif val_lower == 'no_answer':
        return 'no_answer'
    elif val_lower == '':
        return np.nan
    else:
        return np.nan

hfias_cols = [f'hfias_{i}' for i in range(1, 10)]
hfias_invalid_count = 0

for col in hfias_cols:
    if col in df.columns:
        before_valid = df[col].notna().sum()
        df[col] = df[col].apply(clean_hfias_string)
        after_valid = df[col].notna().sum()
        invalid = before_valid - after_valid
        hfias_invalid_count += invalid
        if invalid > 0:
            print(f"  {col}: {invalid} invalid entries set to NaN | Valid: {after_valid}")

fies_cols = [f'fies_{i}' for i in range(1, 9)]
fies_invalid_count = 0

for col in fies_cols:
    if col in df.columns:
        before_valid = df[col].notna().sum()
        df[col] = df[col].apply(clean_fies_string)
        after_valid = df[col].notna().sum()
        invalid = before_valid - after_valid
        fies_invalid_count += invalid
        if invalid > 0:
            print(f"  {col}: {invalid} invalid entries set to NaN | Valid: {after_valid}")

In [54]:
income_cols = [
    'crop_income_per_year_1', 'crop_income_per_year_2', 'crop_income_per_year_3',
    'livestock_sale_income_1', 'livestock_sale_income_2'
]

harvest_cols = [
    'crop_harvest_kg_per_year_1', 'crop_harvest_kg_per_year_2', 'crop_harvest_kg_per_year_3'
]

for col in income_cols:
    if col in df.columns:
        before_valid = df[col].notna().sum()
        df[col] = pd.to_numeric(df[col], errors='coerce')
        after_valid = df[col].notna().sum()
        invalid = before_valid - after_valid
        if invalid > 0:
            print(f"  {col}: {invalid} non-numeric entries → NaN | Valid: {after_valid}")

# Convert harvest columns to numeric
for col in harvest_cols:
    if col in df.columns:
        before_valid = df[col].notna().sum()
        df[col] = pd.to_numeric(df[col], errors='coerce')
        after_valid = df[col].notna().sum()
        invalid = before_valid - after_valid
        if invalid > 0:
            print(f"  {col}: {invalid} non-numeric entries → NaN | Valid: {after_valid}")


# Clean offfarm_income_proportion as categorical
offfarm_prop_ok = {"little", "underhalf", "half", "most", "all", "none"}
df['offfarm_income_proportion'] = df['offfarm_income_proportion'].apply(
    lambda v: v if pd.notna(v) and str(v).strip().lower() in offfarm_prop_ok else np.nan
)


---
## 5. Export Processed Data
Export the cleaned dataset. HFIAS source strings, `unitland`, and `landcultivated` are all retained. No intermediate columns are dropped. Canonical cleaning logic is in `scripts/etl_pipeline.py`.


In [ ]:
columns_for_export = [
    'id_unique', 'country', 'iso_country_code', 'year', 'region',
    'gps_lat_rounded', 'gps_lon_rounded',
    'respondentsex', 'age_malehead', 'age_femalehead', 'education_level', 'respondent_is_head',
    'children_under_4', 'children_4to10', 'males11to24', 'females11to24',
    'males25to50', 'females25to50', 'malesover50', 'femalesover50',
    'count_people', 'household_size_derived',
    'landcultivated', 'unitland', 'land_cultivated_ha', 'land_ownership', 'land_tenure', 'land_irrigated', 'farm_labour',
    'crop_count', 'crop_name_1', 'crop_name_2', 'crop_name_3', 'crop_name_4', 'crop_name_5',
    'crop_harvest_kg_per_year_1', 'crop_harvest_kg_per_year_2', 'crop_harvest_kg_per_year_3',
    'crop_income_per_year_1', 'crop_income_per_year_2', 'crop_income_per_year_3',
    'crop_land_area_1', 'crop_land_area_2', 'crop_land_area_3',
    'crop_consumed_prop_1', 'crop_consumed_prop_2', 'crop_consumed_prop_3',
    'local_currency', 'offfarm_incomes_any', 'offfarm_income_proportion',
    'livestock_sale_income_1', 'livestock_sale_income_2',
    'hfias_1', 'hfias_2', 'hfias_3', 'hfias_4', 'hfias_5',
    'hfias_6', 'hfias_7', 'hfias_8', 'hfias_9',
    'fies_1', 'fies_2', 'fies_3', 'fies_4', 'fies_5',
    'fies_6', 'fies_7', 'fies_8',
    'foodshortagetime',
    # Gender & Resource Control
    'land_ownership',
    'crop_who_control_revenue_1', 'crop_who_control_revenue_2', 'crop_who_control_revenue_3',
    'offfarm_who_control_revenue_1', 'offfarm_who_control_revenue_2',
    'livestock_meat_who_control_eating_1', 'dairy_products_who_control_eating',
]
columns_for_export = [col for col in dict.fromkeys(columns_for_export) if col in df.columns]
df_final = df[columns_for_export].copy()
processed_path = '../data/processed/rhomis_cleaned.csv'
df_final.to_csv(processed_path, index=False)
# Also export compressed version
df_final.to_csv('../data/processed/rhomis_cleaned.csv.gz', index=False, compression='gzip')


---
## 6. Imputation Policy & Justification

**Foundational Principle:** Do not impute. Leave missing values as NaN and document the missing rate clearly.

The RHoMIS dataset has two types of missing data:


1. **Structurally missing (MNAR):** Question not asked in that survey wave (e.g., HFIAS not deployed everywhere)
2. **Conditionally missing (MAR):** Missing depends on another variable (e.g., crop income missing for non-crop households)

### Decision: No Imputation

| Column | Missing % | Decision | Reason |
|---|---|---|---|
| `count_people` | 49.4% | No impute | MNAR; derive from age-band sum (0% missing) |
| `age_malehead` / `age_femalehead` | 65% / 62% | No impute | MNAR; 35–38% coverage is sufficient; missingness is structural |
| `land_cultivated_ha` | 3.2% | No impute | Leave NaN; 3.2% exclusion acceptable |
| `crop_income_per_year_*` | 12.5%–60% | No impute | Never fill with 0; missing ≠ zero income |
| `livestock_sale_income_*` | 22.5%–45% | No impute | Same as above |
| `hfias_1–9` | 69–87% | No impute | Structurally missing; defines food-security sub-population |
| `fies_1–8` | 54.2% | No impute | Structurally missing; most complete FS instrument (~25k rows) |

### Examples of Imputation Decisions Made in This Notebook

- **count_people:** Not filled. Instead, `household_size_derived` (from age bands) provides reliable proxy with 0% missing.
- **Age columns:** Fixed birth-year contamination first; left remaining NaN intact. At 65% missing, imputation indefensible.
- **Income:** Converted to numeric but left NaN as-is. No median/mean fill (missing ≠ zero).
- **Land area:** Unit converted; unmappable units → NaN. Not filled.
- **HFIAS/FIES:** Cleaned strings; not scored or imputed. Analysis will treat as sub-populations.
